In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 4 — Evaluation and comparison (R4)

**Scope:** canonical folds for the grid searches, shared metric code, `comparativa.csv` and the paired per-fold differences of P4.5.

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
> - Every number written in a Markdown cell must appear in a printed output of the notebook.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`, also passed as `cv=folds`), `CLASS_NAMES` (cyber).

## Setup

In [2]:
import numpy as np
import pandas as pd
import json
from sklearn import metrics


# Challenge A — cyber

## Setup from R1 (cyber)

R1's data cell (Fernando), copied without changes. It is **not** pasted again into the final notebook.

In [3]:
# Copied from R1 (Fernando), unchanged: the data folders used by R1's cells
CYBER_DIR = "../datos/cyber"
STORAGE_DIR = "../datos/storage"
FOLDS_DIR = "../folds"

In [4]:
# Load the tables and the class codes without modifying the original files.
dev = pd.read_csv(CYBER_DIR + "/cyber_development.csv")
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")
cyber_folds = pd.read_csv(FOLDS_DIR + "/cyber_folds.csv")

# The JSON keys are text; convert them to the integer codes used in y.
with open(CYBER_DIR + "/cyber_class_codes.json", encoding="utf-8") as file:
    class_codes = json.load(file)

CLASS_NAMES = {}
for code, name in class_codes.items():
    CLASS_NAMES[int(code)] = name

# The 11 source predictors of P2. Metadata columns are never predictors.
NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

expected_columns = ["row_id"] + FEATURES + ["attack_code"]
assert list(dev.columns) == expected_columns, "Unexpected cyber schema"

X = dev[FEATURES].copy()
y = dev["attack_code"].copy()
for column in ["row_id", "feature_group", "config_group", "fold_id", "attack_code"]:
    assert column not in X.columns, "Forbidden predictor: " + column

assert X.shape[1] == 11, "Expected 11 source predictors"
assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert pd.api.types.is_integer_dtype(y), "Class codes must be integers"
assert sorted(y.unique()) == sorted(CLASS_NAMES), "Class codes do not match JSON"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Class names:", CLASS_NAMES)

Development table: (34928, 13)
Predictors: (34928, 11) ( 3 nominal and 8 numeric )
Class names: {0: 'backdoor', 1: 'ddos', 2: 'dos', 3: 'injection', 4: 'mitm', 5: 'normal', 6: 'password', 7: 'ransomware', 8: 'scanning', 9: 'xss'}


## Canonical folds (publish for R2)

In [5]:
fold_table = pd.read_csv("../folds/cyber_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx,val_idx))

print(len(folds))



5


In [6]:
grps = cyber_groups["feature_group"].to_numpy()
#the same for the groups file
assert np.array_equal(cyber_groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the patterns are in different groups in train and in validation
for train_idx, val_idx in folds:
    grps_train = grps[train_idx]
    grps_train = set(grps_train)
    grps_val = grps[val_idx]
    grps_val = set(grps_val) #we created two different sets to see the distinct groups in each one
    assert(len(grps_train & grps_val) == 0) #here we check that the intersection between the two groups is 0
    print(len(train_idx),len(val_idx))

print("no feature group is in both training and validation")




27942 6986
27942 6986
27943 6985
27942 6986
27943 6985
no feature group is in both training and validation


In [7]:
def spread(values):
    res = np.max(values) - np.min(values)
    
    return res

def cyber_metrics(y_true, y_pred):
    macro_f1 = metrics.f1_score(y_true,y_pred,average="macro")
    weighted_f1 = metrics.f1_score(y_true,y_pred,average="weighted")
    accuracy = metrics.accuracy_score(y_true,y_pred)


    return [macro_f1, weighted_f1, accuracy]


In [8]:
#check: the spread of the P2 KNN must be 0.0062, as in comparativa.csv
print(round(spread([0.9436, 0.9473, 0.9490, 0.9428, 0.9451]), 4))

0.0062


## Paired comparison against Practice 3 and Practice 2

In [9]:
# five per-fold primary scores of one earlier row of comparativa.csv
def previous_scores(milestone, challenge, family):
    table = pd.read_csv("../comparativa.csv")
    # keep only the row with this milestone, challenge and family
    row = table[(table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family)]
    # the five scores are stored as one text, e.g. "0.9436 0.9473 0.9490 0.9428 0.9451"
    text = row["primary_metric_per_fold"].iloc[0]
    # cut the text at the spaces and turn each piece into a number
    return np.array(text.split(), dtype=float)




p2_cyber = previous_scores("P2", "cyber", "KNN")
p3_cyber = previous_scores("P3", "cyber", "LogisticRegression")
print("P2 KNN:   ", p2_cyber)
print("P3 logreg:", p3_cyber)

# check of the method with rows we already have: P3 - P2 must give
# the differences printed in Practice 3 (mean -0.1074)
deltas = p3_cyber - p2_cyber
print("P3 - P2:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# R2's five macro F1 of the selected tree (from R2's notebook, printed there with 6 decimals)
# delta = tree - previous: a positive delta means the tree did better on that fold
tree_cyber = np.array([0.945324, 0.949640, 0.949863, 0.941546, 0.942337])

p3_diff = tree_cyber - p3_cyber
p2_diff = tree_cyber - p2_cyber

print("tree:     ", np.round(tree_cyber, 4), "| mean:", round(np.mean(tree_cyber), 4), "| spread:", round(spread(tree_cyber), 4))
print()
print("tree - P3:", np.round(p3_diff, 4))
print("mean:", round(np.mean(p3_diff), 4), "| spread:", round(spread(p3_diff), 4), "| folds where the tree is better:", np.sum(p3_diff > 0), "of 5")
print("spread of P3 logreg:", round(spread(p3_cyber), 4))
print()
print("tree - P2:", np.round(p2_diff, 4))
print("mean:", round(np.mean(p2_diff), 4), "| spread:", round(spread(p2_diff), 4), "| folds where the tree is better:", np.sum(p2_diff > 0), "of 5")
print("spread of P2 KNN:", round(spread(p2_cyber), 4))

P2 KNN:    [0.9436 0.9473 0.949  0.9428 0.9451]
P3 logreg: [0.8407 0.8332 0.8453 0.8353 0.8361]
P3 - P2:   [-0.1029 -0.1141 -0.1037 -0.1075 -0.109 ]
mean: -0.1074 | spread: 0.0112
tree:      [0.9453 0.9496 0.9499 0.9415 0.9423] | mean: 0.9457 | spread: 0.0083

tree - P3: [0.1046 0.1164 0.1046 0.1062 0.1062]
mean: 0.1076 | spread: 0.0119 | folds where the tree is better: 5 of 5
spread of P3 logreg: 0.0121

tree - P2: [ 0.0017  0.0023  0.0009 -0.0013 -0.0028]
mean: 0.0002 | spread: 0.0051 | folds where the tree is better: 3 of 5
spread of P2 KNN: 0.0062


**Reading of the differences, and which kind of comparison this is.**

*Against the Practice 3 logistic model.* The tree is better on all five folds. The mean paired difference is +0.1076 in macro F1, with a range (max − min) of 0.0119 across the five differences. The fold-to-fold range of each model is small next to this change (0.0083 for the tree and 0.0121 for the logistic model), so the change is large relative to the variability we observed.

*Against the Practice 2 KNN.* The two models are practically level. The mean paired difference is +0.0002, the tree is better on three folds and worse on two, and the differences go from −0.0028 to +0.0023. This change is small next to the fold-to-fold range of either model (0.0083 for the tree and 0.0062 for the KNN).

Both readings describe these five folds only. They are not a significance test, and they do not establish that a difference would hold on new data.

*Which kind of comparison.* Both are **pipeline comparisons**, not isolated family comparisons, because the representation changed. The tree keeps every one-hot level and uses the numeric predictors unchanged. The Practice 3 model dropped one reference level per nominal predictor and used `log1p` and `StandardScaler`. The Practice 2 KNN kept every level, like the tree, but also used `log1p` and `StandardScaler`. For a tree, `log1p` and scaling are increasing transformations: they move each threshold without changing which rows fall on each side, so they would not change the tree's partitions. Even so, we did not rerun the earlier families under the tree's representation, so we only claim that this pipeline did better than (P3) or about the same as (P2) the earlier pipelines on these folds.

# Challenge B — storage

## Setup from R1 (storage)

R1's data cell (Fernando), copied without changes. It is **not** pasted again into the final notebook.

In [10]:
# Same names as in cyber, so that R5 can join the two blocks.
dev = pd.read_csv(STORAGE_DIR + "/storage_development.csv")
storage_groups = pd.read_csv(STORAGE_DIR + "/storage_groups.csv")
storage_folds = pd.read_csv(FOLDS_DIR + "/storage_folds.csv")

# raid is a nominal label, even though its levels contain numbers.
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

# The CSV has a different column order: compare the sorted names.
expected_columns = ["row_id"] + FEATURES + ["latency_ns"]
assert sorted(dev.columns) == sorted(expected_columns), "Unexpected storage schema"
assert "iops" not in dev.columns, "Excluded column iops must not be present"
X = dev[FEATURES].copy()
latency_ns = dev["latency_ns"].copy()

assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert np.isfinite(latency_ns).all(), "Non-finite target values"
assert (latency_ns > 0).all(), "Latency must be strictly positive"
z = np.log10(latency_ns)

for column in ["row_id", "config_group", "feature_group", "fold_id", "latency_ns", "iops"]:
    assert column not in X.columns, "Forbidden predictor: " + column
assert X.shape[1] == 9, "Expected nine source predictors"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Target: z = log10(latency_ns)")

Development table:

 (12190, 11)
Predictors: (12190, 9) ( 4 nominal and 5 numeric )
Target: z = log10(latency_ns)


## Canonical folds and metrics (publish for R3)

In [11]:
fold_table = pd.read_csv("../folds/storage_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds (this list is what R3 passes as cv=folds)
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx, val_idx))

print(len(folds))

5


In [12]:
grps = storage_groups["config_group"].to_numpy()
#the same for the groups file
assert np.array_equal(storage_groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the 5 replicates of a configuration never end up split between train and validation
for train_idx, val_idx in folds:
    grps_train = set(grps[train_idx])
    grps_val = set(grps[val_idx])
    assert len(grps_train & grps_val) == 0 #the two sets of groups must not share any group
    print(len(train_idx), len(val_idx))

print("no config group is in both training and validation")

9750 2440
9750 2440
9750 2440
9755 2435
9755 2435
no config group is in both training and validation


In [13]:
#storage metrics on the validation rows of one fold
#the first three are computed on z; MAE_ns only after turning z_pred back into nanoseconds
def storage_metrics(z_true, z_pred, latency_true):
    mae = metrics.mean_absolute_error(z_true, z_pred)
    rmse = np.sqrt(metrics.mean_squared_error(z_true, z_pred))
    r2 = metrics.r2_score(z_true, z_pred)
    mae_ns = metrics.mean_absolute_error(latency_true, 10 ** z_pred)

    return [mae, rmse, r2, mae_ns]

## Paired comparison against Practice 3 and Practice 2

In [14]:
p2_storage = previous_scores("P2", "storage", "KNN")
p3_storage = previous_scores("P3", "storage", "LinearRegression")
print("P2 KNN:   ", p2_storage)
print("P3 linear:", p3_storage)

# check of the method: P2 - P3 must give the differences printed in Practice 3 (mean -0.0181)
# for MAE(z) lower is better, so delta = previous - new
deltas = p2_storage - p3_storage
print("P2 - P3:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# R3's five MAE(z) of the selected tree (from R3's notebook)
# delta = previous - tree: for MAE(z) lower is better, so a positive delta means the tree did better
tree_storage = np.array([0.1238, 0.1286, 0.1319, 0.1353, 0.1332])

p3_diff = p3_storage - tree_storage
p2_diff = p2_storage - tree_storage

print("tree:     ", tree_storage, "| mean:", round(np.mean(tree_storage), 4), "| spread:", round(spread(tree_storage), 4))
print()
print("P3 - tree:", np.round(p3_diff, 4))
print("mean:", round(np.mean(p3_diff), 4), "| spread:", round(spread(p3_diff), 4), "| folds where the tree is better:", np.sum(p3_diff > 0), "of 5")
print("spread of P3 linear:", round(spread(p3_storage), 4))
print()
print("P2 - tree:", np.round(p2_diff, 4))
print("mean:", round(np.mean(p2_diff), 4), "| spread:", round(spread(p2_diff), 4), "| folds where the tree is better:", np.sum(p2_diff > 0), "of 5")
print("spread of P2 KNN:", round(spread(p2_storage), 4))

P2 KNN:   

 [0.1816 0.187  0.1778 0.1775 0.1762]
P3 linear: [0.2047 0.1937 0.1984 0.1953 0.1987]
P2 - P3:   [-0.0231 -0.0067 -0.0206 -0.0178 -0.0225]
mean: -0.0181 | spread: 0.0164
tree:      [0.1238 0.1286 0.1319 0.1353 0.1332] | mean: 0.1306 | spread: 0.0115

P3 - tree: [0.0809 0.0651 0.0665 0.06   0.0655]
mean: 0.0676 | spread: 0.0209 | folds where the tree is better: 5 of 5
spread of P3 linear: 0.011

P2 - tree: [0.0578 0.0584 0.0459 0.0422 0.043 ]
mean: 0.0495 | spread: 0.0162 | folds where the tree is better: 5 of 5
spread of P2 KNN: 0.0108


**Reading of the differences, and which kind of comparison this is.**

*Against the Practice 3 linear regression.* The tree is better on all five folds. The mean paired difference is +0.0676 in MAE(z), with a range of 0.0209 across the five differences. Both models have a much smaller fold-to-fold range (0.0115 for the tree and 0.011 for the linear model), so the change is large relative to the variability we observed.

*Against the Practice 2 KNN.* The tree is also better on all five folds. The mean paired difference is +0.0495, with a range of 0.0162, again large next to the fold-to-fold range of each model (0.0115 for the tree and 0.0108 for the KNN).

Both readings describe these five folds only. They are not a significance test, and they do not establish that a difference would hold on new data.

*Which kind of comparison.* Both are **pipeline comparisons**. The tree keeps every one-hot level and uses the numeric predictors unscaled. The Practice 3 model dropped one reference level per nominal predictor and scaled the numeric predictors; the Practice 2 KNN kept every level but scaled them. Scaling would not change a tree's partitions, but we did not rerun the earlier families under the tree's representation, so the evidence supports only that this pipeline did better than the earlier pipelines on these folds.

# Comparison table

In [15]:
# the two Practice 4 rows, appended at the end of comparativa.csv
# the P2 and P3 rows are never rewritten: mode="a" only adds lines at the end of the file
table = pd.read_csv("../comparativa.csv")

cyber_row = {
    "milestone": "P4", "challenge": "cyber", "family": "DecisionTree", "route": "primary",
    "configuration": "criterion=entropy, max_depth=20, min_samples_leaf=1, ccp_alpha=0.0001; selected by highest mean macro F1 "
                     "from max_depth [3, 5, 8, 12, 20, None], min_samples_leaf [1, 5, 20, 50], ccp_alpha [0, 1e-05, 0.0001, 0.001], criterion [gini, entropy]",
    "preprocessing": "one-hot on the 3 nominal predictors with every level kept; numeric predictors unchanged (no log1p, no scaler); fitted inside each fold",
    "features": "features.md: the 11 distributed source predictors (frozen basis)",
    "validation": "canonical StratifiedGroupKFold(5) on feature_group, read from folds/cyber_folds.csv; grid search with cv=folds",
    "primary_metric_per_fold": " ".join(f"{v:.4f}" for v in tree_cyber),
    "primary_metric": round(np.mean(tree_cyber), 4),
    "primary_metric_spread": round(spread(tree_cyber), 4),
    "supporting_metrics": "weighted F1 0.9625; accuracy 0.9622; per-class precision/recall/F1/support and confusion in p4_cyber.ipynb",
    "fit_time_s": 0.1528,
    "interpretability": "rules from root to leaf; source-level impurity importances (no sign, training fit)",
    "notes": "non-linear model; about equal to the P2 KNN and clearly above the P3 logistic model on every fold; mitm is the weakest class",
}

storage_row = {
    "milestone": "P4", "challenge": "storage", "family": "DecisionTree", "route": "primary",
    "configuration": "criterion=squared_error, max_depth=12, min_samples_leaf=5, ccp_alpha=0.0; selected by lowest mean MAE(z) "
                     "from max_depth [4, 6, 8, 10, 12, None], min_samples_leaf [5, 10, 20, 40], ccp_alpha [0, 1e-05, 0.0001, 0.001]",
    "preprocessing": "one-hot on the 4 nominal predictors with every level kept; numeric predictors unchanged (no scaler); fitted inside each fold; target z = log10(latency_ns)",
    "features": "features.md: the 9 distributed source predictors (frozen basis)",
    "validation": "canonical GroupKFold(5) on config_group, read from folds/storage_folds.csv; grid search with cv=folds",
    "primary_metric_per_fold": " ".join(f"{v:.4f}" for v in tree_storage),
    "primary_metric": round(np.mean(tree_storage), 4),
    "primary_metric_spread": round(spread(tree_storage), 4),
    "supporting_metrics": "RMSE(z) 0.2046; R2(z) 0.9405; MAE_ns 14073675 (computed after inverting predictions)",
    "fit_time_s": 0.058,
    "interpretability": "rules from root to leaf, read on the physical scale with 10^z",
    "notes": "non-linear model with its own grid; better than the P2 KNN and the P3 linear model on every fold",
}

for row in [cyber_row, storage_row]:
    # the row must have exactly the columns of the table, in the same order
    assert list(row.keys()) == list(table.columns)
    # if the notebook is run again, do not add the same row twice
    already = (table["milestone"] == row["milestone"]) & (table["challenge"] == row["challenge"]) & (table["family"] == row["family"])
    if already.any():
        print("already in comparativa.csv:", row["milestone"], row["challenge"], row["family"])
    else:
        pd.DataFrame([row]).to_csv("../comparativa.csv", mode="a", header=False, index=False)
        print("added to comparativa.csv:", row["milestone"], row["challenge"], row["family"])

# the whole table after the append: P2 and P3 rows unchanged, P4 rows at the end
final_table = pd.read_csv("../comparativa.csv")
print(final_table[["milestone", "challenge", "family", "route", "primary_metric", "primary_metric_spread"]].to_string(index=False))

added to comparativa.csv: P4 cyber DecisionTree
added to comparativa.csv: P4 storage DecisionTree
milestone challenge             family   route  primary_metric  primary_metric_spread
       P2     cyber              Dummy primary          0.0242                 0.0000
       P2   storage              Dummy primary          0.6770                 0.0294
       P2     cyber                KNN primary          0.9456                 0.0062
       P2   storage                KNN primary          0.1800                 0.0109
       P3     cyber LogisticRegression primary          0.8381                 0.0121
       P3   storage   LinearRegression primary          0.1981                 0.0110
       P4     cyber       DecisionTree primary          0.9457                 0.0083
       P4   storage       DecisionTree primary          0.1306                 0.0115


## Penalty check (R4 veto)

We checked the R2 and R3 notebooks against the automatic penalties of the statement.

| Check | R2, cyber (final version, in `main`) | R3, storage (draft in `r3-storage`, not final yet) |
|---|---|---|
| Grid search and curves on the canonical folds (`cv=folds`, never `cv=5`) | yes | yes |
| Folds read from `folds/`, not rebuilt | yes | yes |
| Encoder fitted inside each fold, in a `Pipeline`; no scaler | yes | yes |
| Grouping metadata or `iops` used as a feature | no | no |
| Selection with an extra held-out partition | no | no |
| Classification hyperparameters transplanted to regression | not applicable | no: its own grid, selected by MAE(z) |
| MAE in nanoseconds computed after `10 ** z_hat`, never by exponentiating the MAE | not applicable | yes |
| A difference called significant, or seeds compared | no | no |

No penalty was found. The R3 check must be repeated on its final version before the submission.